# 🔧 Meme & Sarcasm Understanding – Preprocessing Pipeline

This notebook demonstrates the complete preprocessing pipeline:
1. Text cleaning steps (with before/after comparison)
2. Image preprocessing (resize, normalize, augment)
3. Tokenization demo
4. Dataset splitting and loading
5. DataLoader verification

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from PIL import Image
from torchvision import transforms

from preprocessing import (
    clean_text, remove_html, remove_urls, remove_emojis,
    expand_hashtags, remove_stopwords, generate_sample_dataset
)
from data_loader import SampleDatasetLoader, get_transform

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print('Imports successful ✓')

## 1. Text Cleaning Pipeline – Step-by-Step

In [ ]:
sample_texts = [
    'Oh sure!! 😂😂 Because <b>Mondays</b> are TOTALLY awesome!! #MondayBlues https://t.co/xyz',
    'WOW @user1 @user2 great job!! Just LOVE when the meeting runs 3 hours 🙄🙄 #productivity',
    'Beautiful sunrise today ☀️ Feeling grateful for another day to learn #blessed',
]

pipeline_steps = [
    ('Original',             lambda t: t),
    ('Remove HTML',          remove_html),
    ('Remove URLs',          remove_urls),
    ('Expand Hashtags',      expand_hashtags),
    ('Remove Emojis',        remove_emojis),
    ('Full clean_text()',    clean_text),
    ('+ Remove Stopwords',   lambda t: remove_stopwords(clean_text(t))),
]

for text in sample_texts:
    print('\n' + '='*65)
    for step_name, fn in pipeline_steps:
        result = fn(text)
        print(f"  {step_name:<22}: {result[:70]}")

print('\nText cleaning pipeline demonstration complete ✓')

## 2. Image Preprocessing & Augmentation

In [ ]:
# Ensure sample dataset exists
if not Path('../data/sample_dataset/metadata.json').exists():
    generate_sample_dataset(n_samples=200)

with open('../data/sample_dataset/metadata.json') as f:
    samples = json.load(f)

img_path = Path('../data/sample_dataset/images') / samples[0]['image_file']
raw_img  = Image.open(img_path).convert('RGB')

# Demonstrate various transforms
transforms_demo = [
    ('Original (224×224)',   transforms.Compose([transforms.Resize((224,224))])),
    ('H-Flip',               transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(p=1.0)])),
    ('Color Jitter',         transforms.Compose([transforms.Resize((224,224)), transforms.ColorJitter(0.4,0.4,0.3,0.1)])),
    ('Random Crop',          transforms.Compose([transforms.Resize((256,256)), transforms.RandomCrop(224)])),
    ('Grayscale',            transforms.Compose([transforms.Resize((224,224)), transforms.Grayscale(3)])),
]

fig, axes = plt.subplots(1, len(transforms_demo), figsize=(18, 4))
for ax, (name, tfm) in zip(axes, transforms_demo):
    ax.imshow(tfm(raw_img))
    ax.set_title(name, fontsize=10)
    ax.axis('off')

plt.suptitle('Image Augmentation Strategies', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/graphs/preprocessing_augmentations.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Normalization Verification

In [ ]:
# Verify ImageNet normalization
to_tensor = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
normalized = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

raw_t = to_tensor(raw_img)
nor_t = normalized(raw_img)

print('Before normalization:')
print(f'  Min={raw_t.min():.3f}  Max={raw_t.max():.3f}  Mean={raw_t.mean():.3f}')
print('After ImageNet normalization:')
print(f'  Min={nor_t.min():.3f}  Max={nor_t.max():.3f}  Mean={nor_t.mean():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (t, title) in zip(axes, [(raw_t, 'Before Normalize'), (nor_t.clamp(0,1), 'After Normalize (clamped)')]):
    ax.imshow(t.permute(1,2,0).numpy())
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.savefig('../outputs/graphs/preprocessing_normalization.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. DataLoader Verification

In [ ]:
loader    = SampleDatasetLoader(batch_size=8)
train_dl, val_dl, test_dl = loader.get_loaders()

batch = next(iter(train_dl))
print('Batch contents:')
for key, val in batch.items():
    if isinstance(val, torch.Tensor):
        print(f'  {key:<16}: shape={val.shape}  dtype={val.dtype}')

print(f'\nNumber of batches – Train={len(train_dl)}  Val={len(val_dl)}  Test={len(test_dl)}')
print(f'Labels in batch: {batch["label"].tolist()}')
print('DataLoader verification complete ✓')

## 5. Save Processed Metadata

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

df = pd.DataFrame(samples)
df['clean_text'] = df['text'].apply(clean_text)
df['text_len']   = df['text'].apply(lambda x: len(x.split()))
df.to_csv('../data/processed/sample_cleaned.csv', index=False)

print('Processed metadata saved to data/processed/sample_cleaned.csv')
print(df.head())